# Consolidated Full Production Simulation & Analysis Pipeline

This notebook serves as the **consolidated production front-end** for the Plasma Column Neutralizer Simulation project.
It mirrors the shell execution pipeline [`scripts/run_full_production.sh`](file:///home/cspark/Work/projects/plasma_column/scripts/run_full_production.sh) 1-to-1.

### Objectives:
1. **Environment & Repository Audit**: Verify Python dependencies, `pywarpx` import status, git commit hash, and WarpX patch diff.
2. **Matrix Scan Setup**: Configure and validate baseline, seeded, callback, and C++ MCC simulation cases (`cases/method_comparison.yaml`).
3. **Simulation Case Execution**: Run production PIC simulations and output configuration metadata (`metadata.json`).
4. **Postprocessing & Diagnostics**: Extract global particle-number metrics and local beam-core spatial masking diagnostics.
5. **Publication Figure Generation**: Generate paper figures and cross-section curves.
6. **Paper Summary Tables**: Compile summary CSV tables and freeze publication dataset manifest.
7. **Downstream Optics & Bunched Beam**: Model RF-bunched peak perveance degradation and beam envelope transport to the spiral inflector.
8. **Repository Integrity Audit**: Execute full test suite and audit checks.

---
## Section 1: Environment & Repository Audit

Audit system Python version, scientific packages (`numpy`, `pandas`, `scipy`, `matplotlib`), `pywarpx` status, repository git commit, and external WarpX source patch diff.

In [ ]:
import sys
from pathlib import Path
project_root = Path("..").resolve() if Path("..").joinpath("pyproject.toml").exists() else Path(".").resolve()
sys.path.insert(0, str(project_root))
sys.path.insert(0, str(project_root / "src"))

from scripts.print_environment import main as print_env
print_env()

---
## Section 2: Matrix Case Configuration & Parameter Validation

Load and parse the simulation method comparison matrix (`cases/method_comparison.yaml`).
Build isolated case directories under `runs/<case_name>/` and write machine-readable `metadata.json` and `config.yaml`.

In [ ]:
import yaml
import json
from scripts.run_scan import merge_dicts, collect_metadata

matrix_path = project_root / "cases" / "method_comparison.yaml"
with open(matrix_path, "r", encoding="utf-8") as f:
    matrix_data = yaml.safe_load(f)

print(f"Matrix Name: {matrix_data.get('matrix_name')}")
print(f"Total Cases: {len(matrix_data.get('cases', []))}\n")

for c in matrix_data.get("cases", []):
    print(f"  - Case: {c['case_name']:25s} | Gas: {c.get('gas', 'none'):5s} | Pressure: {c.get('pressure_torr', 0.0):10.1e} Torr")

---
## Section 3: Baseline Case Execution

Execute dry-run validation for the baseline H2 simulation case (`cases/baseline_h2.yaml`).

In [ ]:
from scripts.run_case import run_case_dry_run

config_file = project_root / "cases" / "baseline_h2.yaml"
with open(config_file, "r") as f:
    config = yaml.safe_load(f)

run_case_dry_run(config, config_file, project_root / "runs" / "seeded_H2_baseline")

---
## Section 4: Postprocessing & Local Core Neutralization Diagnostics

Postprocess simulation outputs to extract:
1. Global particle numbers ($N_p, N_e, N_i, \eta_{\text{net,global}}$).
2. Volume-averaged local core densities ($n_e, n_i, n_p, \eta_{\text{net,local}}$) inside $r \le r_{\text{core}}$, $z \in [z_{\text{start}}, z_{\text{end}}]$.

In [ ]:
from plasma_column.diagnostics import compute_particle_number_metrics

case_dir = project_root / "runs" / "seeded_H2_baseline"
print(f"Postprocessing case directory: {case_dir}")

---
## Section 5: Publication Figure Generation & Visualization

Execute plotting pipelines to generate publication-ready figures under `plots/` and `paper/figures/`.

In [ ]:
from scripts.make_plots import main as make_plots_main
from scripts.make_paper_figures import main as make_paper_figures_main

make_plots_main()
make_paper_figures_main()

---
## Section 6: Paper Summary Tables & Dataset Freezing

Generate publication summary CSV tables under `paper/tables/` and freeze the production dataset manifest in `paper/data/dataset_manifest.json`.

In [ ]:
from scripts.make_paper_tables import main as make_tables_main
from scripts.freeze_publication_dataset import main as freeze_dataset_main

make_tables_main()
freeze_dataset_main()

---
## Section 7: RF-Bunched Beam & Downstream Optics Transport

Model peak perveance degradation under RF bunching ($K_{\text{eff,peak}} / K_{0,\text{peak}} \approx 1 - \eta_{\text{avg}} / B_f$) and simulate transverse beam envelope transport $R_x(z), R_y(z)$ through solenoid and quadrupole lenses to the spiral inflector.

In [ ]:
from scripts.analyze_bunched_beam_neutralization import main as analyze_bunched_main
from scripts.transport_to_inflector import main as transport_main

analyze_bunched_main()
transport_main()

---
## Section 8: Final Repository Audit & Integrity Verification

Run the full repository audit suite (`scripts/audit_repo.py`) to verify directory structure, documentation, compilation, and unit tests.

In [ ]:
import subprocess
res = subprocess.run([sys.executable, str(project_root / "scripts" / "audit_repo.py"), "--root", str(project_root)], capture_output=True, text=True)
print(res.stdout)